# Alarm clock: the fallback

**Follow the guide, not this notebook:** <https://rocketride-org.github.io/rocketride-workshops/modules/alarm-clock/>. In it you install RocketRide, run `rocketride init`, create the app, ask Claude Code for the clock, watch it in the App Builder's preview, and deploy it from the App Builder.

This notebook only holds the prompt and a finished clock for anyone who gets stuck. Save it in your project folder, next to `apps/`.

## The prompt

Run `claude` in your project folder's terminal and ask:

> Build a basic analog alarm clock in apps/alarm_clock: a ticking clock face with hour, minute and second hands, the digital time under it, a time picker and a Set alarm button. When the alarm time comes, flash the face and beep until I press Stop. Replace the template README with a short description of the clock. Don't deploy it.

## Stuck?

No Claude Code, or the clock it built won't compile? Run the cell below. It writes a finished clock, and the App Builder's preview picks it up as soon as it's saved.

In [ ]:
# Skip ahead: run this only if the clock Claude built doesn't work, or you have no Claude Code.
# Writes a finished clock into apps/alarm_clock/src/App.tsx, and the README for the app's store listing.
from pathlib import Path

APP_ROOT = Path("apps/alarm_clock")
if not (APP_ROOT / "package.json").exists():
    raise RuntimeError("No apps/alarm_clock here. Save this notebook in your project folder, "
                       "and create the app first (Step 3 of the guide).")

APP_TSX = r"""/**
 * Alarm Clock: an analog clock with one alarm, rendered by the RocketRide shell.
 */

import React, { useEffect, useRef, useState } from 'react';
import { AppLayout } from 'shell';

const SIZE = 260;
const MID = SIZE / 2;

type HandProps = { turns: number; length: number; width: number; color: string };

/** One hand. `turns` is how far round the face it points: 0 is 12 o'clock, 0.5 is 6. */
const Hand: React.FC<HandProps> = ({ turns, length, width, color }) => {
	const angle = turns * 2 * Math.PI;
	const x = MID + length * Math.sin(angle);
	const y = MID - length * Math.cos(angle);
	return <line x1={MID} y1={MID} x2={x} y2={y} stroke={color} strokeWidth={width} strokeLinecap="round" />;
};

/** The face: 60 ticks, 12 numerals, and the three hands. */
const Face: React.FC<{ now: Date; flash: boolean }> = ({ now, flash }) => {
	const seconds = now.getSeconds();
	const minutes = now.getMinutes() + seconds / 60;
	const hours = (now.getHours() % 12) + minutes / 60;

	const ticks = [];
	for (let i = 0; i < 60; i++) {
		const angle = (i / 60) * 2 * Math.PI;
		const inner = i % 5 === 0 ? MID - 18 : MID - 10;
		ticks.push(
			<line key={i} stroke="currentColor" strokeWidth={i % 5 === 0 ? 3 : 1}
				x1={MID + inner * Math.sin(angle)} y1={MID - inner * Math.cos(angle)}
				x2={MID + (MID - 4) * Math.sin(angle)} y2={MID - (MID - 4) * Math.cos(angle)} />,
		);
	}

	const numerals = [];
	for (let n = 1; n <= 12; n++) {
		const angle = (n / 12) * 2 * Math.PI;
		numerals.push(
			<text key={n} x={MID + (MID - 36) * Math.sin(angle)} y={MID - (MID - 36) * Math.cos(angle)}
				textAnchor="middle" dominantBaseline="central" fontSize={18} fill="currentColor">{n}</text>,
		);
	}

	let fill = 'var(--rr-bg-paper, #ffffff)';
	if (flash) {
		fill = 'var(--rr-color-error, #e5484d)';
	}

	return (
		<svg width={SIZE} height={SIZE} viewBox={`0 0 ${SIZE} ${SIZE}`} style={{ color: 'var(--rr-text-primary, #111)' }}>
			<circle cx={MID} cy={MID} r={MID - 2} fill={fill} stroke="currentColor" strokeWidth={3} />
			{ticks}
			{numerals}
			<Hand turns={hours / 12} length={MID * 0.5} width={6} color="currentColor" />
			<Hand turns={minutes / 60} length={MID * 0.75} width={4} color="currentColor" />
			<Hand turns={seconds / 60} length={MID * 0.85} width={2} color="var(--rr-color-error, #e5484d)" />
			<circle cx={MID} cy={MID} r={5} fill="currentColor" />
		</svg>
	);
};

/** Plays one short beep. */
function beep(audio: AudioContext): void {
	const tone = audio.createOscillator();
	tone.frequency.value = 880;
	tone.connect(audio.destination);
	tone.start();
	tone.stop(audio.currentTime + 0.25);
}

const Clock: React.FC = () => {
	const [now, setNow] = useState(new Date());
	const [alarm, setAlarm] = useState('07:00');
	const [isArmed, setIsArmed] = useState(false);
	const [isRinging, setIsRinging] = useState(false);
	const audio = useRef<AudioContext | null>(null);

	// Tick once a second.
	useEffect(() => {
		const timer = setInterval(() => {
			setNow(new Date());
		}, 1000);
		return () => {
			clearInterval(timer);
		};
	}, []);

	// Ring when the clock reaches the alarm time. It fires once, then disarms.
	const hhmm = now.toTimeString().slice(0, 5);
	useEffect(() => {
		if (isArmed && hhmm === alarm) {
			setIsArmed(false);
			setIsRinging(true);
		}
	}, [isArmed, hhmm, alarm]);

	// Beep every second while ringing.
	useEffect(() => {
		if (!isRinging || !audio.current) {
			return;
		}
		const sound = audio.current;
		beep(sound);
		const timer = setInterval(() => {
			beep(sound);
		}, 1000);
		return () => {
			clearInterval(timer);
		};
	}, [isRinging]);

	// Browsers only allow sound after a click, so the audio starts when you set the alarm.
	const toggleAlarm = () => {
		if (!audio.current) {
			audio.current = new AudioContext();
		}
		void audio.current.resume();
		setIsArmed(!isArmed);
	};

	let status = 'Alarm off';
	if (isArmed) {
		status = `Alarm set for ${alarm}`;
	}
	if (isRinging) {
		status = 'Wake up!';
	}

	return (
		<div style={styles.wrap}>
			<Face now={now} flash={isRinging && now.getSeconds() % 2 === 0} />
			<div style={styles.digital}>{now.toLocaleTimeString()}</div>
			<div style={styles.row}>
				<input type="time" value={alarm} onChange={(e) => { setAlarm(e.target.value); }} style={styles.input} />
				<button onClick={toggleAlarm} style={styles.button}>{isArmed ? 'Turn off' : 'Set alarm'}</button>
				{isRinging && <button onClick={() => { setIsRinging(false); }} style={styles.button}>Stop</button>}
			</div>
			<div style={styles.status}>{status}</div>
		</div>
	);
};

const styles: Record<string, React.CSSProperties> = {
	wrap: { display: 'flex', flexDirection: 'column', alignItems: 'center', gap: 16, padding: 32, fontFamily: 'var(--rr-font-family, system-ui)' },
	digital: { fontSize: 28, fontVariantNumeric: 'tabular-nums', color: 'var(--rr-text-primary, #111)' },
	row: { display: 'flex', gap: 8, alignItems: 'center' },
	input: { fontSize: 16, padding: '6px 8px' },
	button: { fontSize: 14, padding: '6px 14px', cursor: 'pointer' },
	status: { fontSize: 13, color: 'var(--rr-text-secondary, #666)' },
};

/** Root view: the shell frame with the clock inside. */
const App: React.FC = () => {
	return (
		<AppLayout showStatus>
			<Clock />
		</AppLayout>
	);
};

export default App;
"""

README_MD = """# Alarm Clock

An analog alarm clock. Pick a time, press **Set alarm**, and when the clock gets there the face flashes and it beeps until you press **Stop**.

Built in the RocketRide alarm-clock workshop.
"""

(APP_ROOT / "src" / "App.tsx").write_text(APP_TSX, encoding="utf-8", newline="\n")
(APP_ROOT / "README.md").write_text(README_MD, encoding="utf-8", newline="\n")
print("Wrote the finished clock. Carry on with Stage 3.")

## What's next

- **Back to the guide:** see the clock in the App Builder, then deploy it from the **Deploy** tab.
- **Docs:** [docs.rocketride.org](https://docs.rocketride.org)
- **Help and community:** [RocketRide Discord](https://discord.gg/PMXrtenMsY)
- **Next module:** [Integrate RocketRide into an existing app](https://rocketride-org.github.io/rocketride-workshops/modules/notepad-integration/)